# 📈 04 Model Training — Linear Regression

## Objective

Train the baseline Linear Regression model using the refined dataset and a group-aware train/test split.

This notebook:

- uses the refined dataset
- excludes `pricepersqft`
- uses `bhk`, `propertytype`, `location`, and `sqft`
- prevents feature-group overlap between train and test
- keeps preprocessing inside the model pipeline
- trains Linear Regression
- saves the complete preprocessing + model pipeline

The canonical trained pipeline is saved for later use.

In [25]:
import joblib
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

In [18]:
df = pd.read_csv(
    "../data/processed/house_prices_refined.csv"
)

print("Dataset shape:", df.shape)

Dataset shape: (13297, 5)


In [26]:
feature_columns = [
    "bhk",
    "propertytype",
    "location",
    "sqft"
]

target_column = "totalprice"

X = df[feature_columns].copy()
y = df[target_column].copy()

print("Features:", feature_columns)
print("Target:", target_column)

Features: ['bhk', 'propertytype', 'location', 'sqft']
Target: totalprice


In [27]:
groups = (
    df[feature_columns]
    .astype(str)
    .agg("||".join, axis=1)
)

print("Total rows:", len(df))
print("Unique feature groups:", groups.nunique())

Total rows: 13297
Unique feature groups: 9359


In [28]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (10556, 4)
X_test shape: (2741, 4)
y_train shape: (10556,)
y_test shape: (2741,)


In [29]:
train_groups = set(groups_train)
test_groups = set(groups_test)

overlap = train_groups.intersection(test_groups)

print("Unique feature groups in train:", len(train_groups))
print("Unique feature groups in test:", len(test_groups))
print("Overlapping feature groups:", len(overlap))

assert len(overlap) == 0, "Feature-group leakage detected!"

print("No feature-group overlap detected.")

Unique feature groups in train: 7487
Unique feature groups in test: 1872
Overlapping feature groups: 0
No feature-group overlap detected.


In [30]:
categorical_features = [
    "propertytype",
    "location"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [31]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

In [32]:
model.fit(
    X_train,
    y_train
)

print("Linear Regression model trained successfully.")

Linear Regression model trained successfully.


In [33]:
predictions = model.predict(X_test)

print("Predictions generated:", len(predictions))

Predictions generated: 2741


In [34]:
results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": predictions
})

results.head(10)

,Actual,Predicted
0,20200000,1.913820e+07
1,10400000,1.066236e+07
2,18500000,1.263495e+07
3,42300000,3.774546e+07
4,27700000,3.137136e+07
5,5270000,1.088812e+07
6,28000000,3.137136e+07
7,11299999,3.152027e+07
8,15000000,2.707980e+07
9,7759999,1.066236e+07


In [35]:
joblib.dump(
    model,
    "../models/house_price_pipeline.pkl"
)

print("Complete model pipeline saved successfully.")

Complete model pipeline saved successfully.


In [36]:
loaded_model = joblib.load(
    "../models/house_price_pipeline.pkl"
)

loaded_predictions = loaded_model.predict(X_test)

print("Saved pipeline loaded successfully.")
print("Predictions match:",
      (predictions == loaded_predictions).all())

Saved pipeline loaded successfully.
Predictions match: True


## Conclusion

The Linear Regression baseline was trained using:

- the refined dataset
- `bhk`, `propertytype`, `location`, and `sqft`
- `totalprice` as the target
- group-aware train/test splitting
- zero feature-group overlap
- preprocessing integrated into the pipeline

The complete preprocessing + model pipeline was saved to:

`../models/house_price_pipeline.pkl`

Model evaluation is handled separately in Notebook 05.
Model comparison is handled separately in Notebook 06.